In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import *
import pyspark.sql.functions as F


## Data Reading JSON


In [0]:
df_json = spark.read.format('json').option('inferSchema',True)\
    .option('multiline',False)\
    .load('/Volumes/workspace/default/my_volume/drivers.json')



In [0]:
df_json.display()

### Data Reading

In [0]:
filepath ='/Volumes/workspace/default/my_volume/BigMart Sales.csv'

In [0]:
df = (spark.read.format('csv').option('inferSchema','true').option('header','true').load(filepath))

In [0]:
display(df)


In [0]:
df.printSchema()

## SELECT TRANSFORMATIN

In [0]:
df_select = df.select(col('Item_Identifier'), 'Item_Weight', 'Item_Fat_Content').display()

**ALIAS**

In [0]:
df.select(col('Item_Identifier').alias('ItemID')).display()

In [0]:
df.display()

# Filter

**Scenario 1**

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()


**Scenario 2**

In [0]:
df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).display()

**Scenario 3**

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1', 'Tier 2'))).display()

# WithColumnRenamed() Transaformation

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

# withColumn() Transformations
**We can add a new columnd and modify existing column value with this.**

In [0]:
#1
df = df.withColumn('new_col', lit('new'))
df.display()

In [0]:
#2
df.withColumn('multiply', col('Item_MRP')*col('Item_Outlet_Sales')).display()

In [0]:
#3 replace Regular with RG and Low Fat with LF
df.withColumn('Item_Fat_content', regexp_replace(col('Item_Fat_Content'), 'Low Fat', 'LF')) \
    .withColumn('Item_Fat_content', regexp_replace(col('Item_Fat_Content'), 'Regular', 'RG')).display()

### Type Casting with cast() function
**This helps to change the datatype of the column**

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

### Sort/Order By transformation


**Scenario 1**

In [0]:
df.sort(col('Item_Visibility').asc()).display()

**Scenario 2**

In [0]:
df.sort(["Item_Visibility", "Item_Weight"], ascending = [1,0]).display()
#here we can sort multiple columns into Ascending or Descending order based on the requirement.Here  [1,0] treats as a boolean value where one is = True, and 0 = False(Which will display order in descending order)

### Limit Function

In [0]:
df.limit(10).display()

### Drop Function

In [0]:
df.drop('Item_Visibility').display()

In [0]:
#for Multiple Columns Drop
df.drop('Item_Identifier', "Item_Weight").display()

In [0]:
df.display()

## Drop_Duplicates()


In [0]:
df.display()

In [0]:
df.dropDuplicates().display()
print("#Duplicates are dropped")

In [0]:
df.distinct().display()

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()
#This command will drop the duplicates from the Item_type column, works same as Distinct

In [0]:
df.drop_duplicates(subset=['Item_Identifier']).display()

In [0]:
df.drop_duplicates(subset= ["Item_Type", "Item_Identifier"]).display()

In [0]:
df.drop(col("Item_Fat_Content")).display()

In [0]:
df.display()

In [0]:
df.sort(["Item_Visibility", "Item_Weight"], ascending = [1,0]).display()


## Intermediate Transformation

In [0]:
# Intermediate Transformation: Add a new column using withColumn + conditional logic (when/otherwise)
from pyspark.sql.functions import when, col, round

df_transformed = (df
    .withColumn("Visibility_Category",
        when(col("Item_Visibility") < 0.02, "Low")
        .when(col("Item_Visibility") < 0.10, "Medium")
        .otherwise("High")
    )
    .withColumn("Weight_Rounded", round(col("Item_Weight"), 2))
)

df_transformed.display()
# Explanation:
# - withColumn adds (or replaces) a column in the DataFrame.
# - when().otherwise() works like SQL CASE WHEN, letting you bucket values into categories.
# - round() is a built-in SQL function applied within withColumn to clean up numeric precision.
# This is a common intermediate pattern: deriving new columns from existing data using conditional logic.

In [0]:
# Intermediate Transformation: GroupBy aggregation + join back to original DataFrame
# from pyspark.sql.functions import col, sum as _sum, avg, count, round

# Step 1 — Aggregate sales metrics per Item_Type
df_item_summary = (df
    .groupBy("Item_Type")
    .agg(
        sum("Item_Outlet_Sales").alias("Total_Sales"),
        avg("Item_MRP").alias("Avg_MRP"),
        count("Item_Identifier").alias("Item_Count"),
    )
)

# Step 2 — Round numeric columns for readability
df_item_summary = df_item_summary.withColumn("Avg_MRP", round(col("Avg_MRP"), 2))

# Step 3 — Join the summary back to the original DataFrame to enrich each row
#         with its category-level totals (a classic 'broadcast small aggregate' pattern)
df_enriched = (df
    .join(df_item_summary, on="Item_Type", how="left")
    .withColumn("Sales_Contribution_Pct",
        round((col("Item_Outlet_Sales") / col("Total_Sales")) * 100, 2)
    )
)
df_enriched.display()

# Explanation:
# - groupBy().agg() computes per-category summaries (total sales, average MRP, item count).
# - join(..., how='left') merges the aggregate back onto every original row so each
#   item can see its category's totals.
# - A derived column (Sales_Contribution_Pct) shows how much each row contributes to its
#   category — a common enrichment pattern in analytics pipelines.

## # Advanced Transformation: Window functions for ranking + lag/lead + running totals


In [0]:
# Advanced Transformation: Window functions for ranking + lag/lead + running totals
from pyspark.sql.window import Window
from pyspark.sql.functions import col, desc, row_number, rank, dense_rank, lag, lead, sum as _sum, round

# Define a window partitioned by Item_Type, ordered by Item_Outlet_Sales descending
window_spec = Window.partitionBy("Item_Type").orderBy(desc("Item_Outlet_Sales"))

# Define a window for running totals (cumulative sum) within each Item_Type
window_running = Window.partitionBy("Item_Type").orderBy(desc("Item_Outlet_Sales")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_advanced = (df
    .withColumn("Sales_Rank", row_number().over(window_spec))           # Unique 1,2,3... ranking
    .withColumn("Sales_Dense_Rank", dense_rank().over(window_spec))       # No gaps in ranking (1,1,2,3...)
    .withColumn("Prev_Sales", lag("Item_Outlet_Sales").over(window_spec)) # Previous row's sales within partition
    .withColumn("Next_Sales", lead("Item_Outlet_Sales").over(window_spec))# Next row's sales within partition
    .withColumn("Running_Total_Sales", round(_sum("Item_Outlet_Sales").over(window_running), 2))  # Cumulative sum
    .withColumn("Sales_Diff_From_Prev", round(col("Item_Outlet_Sales") - col("Prev_Sales"), 2))  # Difference from previous row
)
df_advanced.display()

# Explanation:
# - Window functions operate over a "window" of rows defined by partitionBy() and orderBy().
# - row_number(): assigns a unique sequential rank (1, 2, 3...) within each partition.
# - rank() vs dense_rank(): rank() leaves gaps after ties (1,1,3); dense_rank() does not (1,1,2).
# - lag()/lead(): access values from the previous/next row — great for period-over-period comparisons.
# - rowsBetween(unboundedPreceding, currentRow): creates a running cumulative total within the partition.
# This is the most powerful transformation tier in PySpark: it enables analytics that would otherwise
# require self-joins or expensive shuffles, all in a single declarative pass.
# Patterns: top-N per category, running totals, moving averages, year-over-year growth, etc.

In [0]:
# Scenario: Daily ETL job to load, clean, transform, and save sales data in Databricks

from pyspark.sql.functions import col, to_date, when, round

# Step 1: Read raw data (simulate with existing DataFrame 'df')
# In reality, would use: spark.read.format("csv").option("header", "true").load("path_to_raw_data")

# Step 2: Data Engineering steps — clean, standardize, transform

df_cleaned = (df
    .dropDuplicates(["Item_Identifier", "Outlet_Identifier"])  # Remove duplicate rows by primary keys
    .withColumn("Item_Fat_Content",
        when(col("Item_Fat_Content") == "LF", "Low Fat")
        .when(col("Item_Fat_Content") == "reg", "Regular")
        .otherwise(col("Item_Fat_Content"))
    )  # Standardize item fat content categories
    .withColumn("Report_Date", to_date(col("Outlet_Establishment_Year").cast("string"), "yyyy"))  # Add report date
    .withColumn("Item_Weight", round(col("Item_Weight"), 2))  # Clean numeric precision on weight
    .withColumn("Is_Heavy", when(col("Item_Weight") > 10, 1).otherwise(0))  # Add flag column
)

# Step 3: Aggregate and enrich with summary metrics
from pyspark.sql.functions import sum as _sum, avg, count

df_summary = (df_cleaned
    .groupBy("Outlet_Identifier")
    .agg(
        _sum("Item_Outlet_Sales").alias("Total_Sales"),
        avg("Item_MRP").alias("Avg_MRP"),
        count("Item_Identifier").alias("Num_Items")
    )
)

# Step 4: Join summary metrics back and save curated dataset
df_final = df_cleaned.join(df_summary, on="Outlet_Identifier", how="left")

# Step 5: Write the final DataFrame out (simulate with display)
df_final.display()
# Typical Data Engineering Workflow:
# - Ingest raw data
# - Deduplicate, standardize columns, create derived features
# - Aggregate summary metrics
# - Enrich the dataset via joins
# - Save curated data for downstream analytics or ML

/Volumes/workspace/default/my_volume/BigMart Sales.csv


In [0]:
# Scenario: Read BigMart Sales CSV, transform by adding a "MRP_Category", and write to results_transformation.csv

from pyspark.sql.functions import when, col

# Step 1: Read CSV into Spark DataFrame
df_scenario = spark.read.format("csv").option("header", "true").option("inferSchema", "true") \
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")

# Step 2: Transformation - bucket Item_MRP into price categories
df_transformed_scenario = df_scenario.withColumn(
    "MRP_Category",
    when(col("Item_MRP") < 80, "Low")
    .when(col("Item_MRP") < 150, "Medium")
    .otherwise("High")
)

# Step 3: Write to CSV (overwrite mode), exclude Spark metadata
df_transformed_scenario.coalesce(1).write.option("header", "true").mode("overwrite") \
    .csv("/Volumes/workspace/default/my_volume/destination/")

In [0]:

# 1. Load the data
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true") \
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")

# 2. Fill nulls in "Item_Weight" with 0 first
df_clean = df.fillna({"Item_Weight": 0})

# 3. Apply the conditional logic on the cleaned data
dr_transformation = df_clean.withColumn("Weight_of_Item",
                     when(col("Item_Weight") > 10, "High")
                     .when(col("Item_Weight") < 10, "Low")
                     .otherwise("Medium")
                 )

# 4. Select and display
df_selected = dr_transformation.select(col("Item_Weight"), col("Weight_of_Item"))

df_selected.display()


In [0]:
df.withColumnRenamed("Item_Fat_Content","Item_Fat").display()


In [0]:
df.display()

In [0]:
# ============================================================================
# 📦 Scenario: Building a Mini Data Pipeline — BigMart Sales Data
# Level: Intermediate → Slightly Advanced
# Goal: Cover core Data Engineering steps in one readable flow
# ============================================================================
#
# Real-world Data Engineering pipeline (ETL = Extract, Transform, Load):
#   1. EXTRACT  → Read raw data from a source (CSV, DB, API, etc.)
#   2. CLEAN    → Handle nulls, duplicates, bad values
#   3. TRANSFORM → Add/derive columns, group, aggregate, join, sort
#   4. ENRICH   → Combine with other data (joins, window functions)
#   5. LOAD     → Write the cleaned & curated data to a destination
#
# ============================================================================

from pyspark.sql.functions import col, when, round, sum as _sum, avg, count, desc

# ---------------------------------------------------------------------------
# STEP 1: EXTRACT — Read the raw CSV
# ---------------------------------------------------------------------------
df = (spark.read
    .format("csv")
    .option("header", "true")          # first row is column names
    .option("inferSchema", "true")      # let Spark guess data types
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)

print("✅ STEP 1 — Raw data loaded. Row count:", df.count())
df.display()

# ---------------------------------------------------------------------------
# STEP 2: CLEAN — Fix data quality issues
# ---------------------------------------------------------------------------
# 2a. Remove exact duplicate rows
df = df.dropDuplicates()

# 2b. Fill nulls: Item_Weight → 0, Outlet_Size → "Unknown"
df = df.fillna({"Item_Weight": 0, "Outlet_Size": "Unknown"})

# 2c. Standardize messy category values (e.g., "LF", "low fat" → "Low Fat")
df = df.withColumn("Item_Fat_Content",
    when(col("Item_Fat_Content").isin("LF", "low fat"), "Low Fat")
    .when(col("Item_Fat_Content").isin("reg"), "Regular")
    .otherwise(col("Item_Fat_Content"))
)

print("✅ STEP 2 — Data cleaned: duplicates removed, nulls filled, categories standardized.")
df.display()

# ---------------------------------------------------------------------------
# STEP 3: TRANSFORM — Derive new columns + aggregate
# ---------------------------------------------------------------------------
# 3a. Add a "Price_Category" column by bucketing Item_MRP
df = df.withColumn("Price_Category",
    when(col("Item_MRP") < 80, "Low")
    .when(col("Item_MRP") < 150, "Medium")
    .otherwise("High")
)

# 3b. Add a "Visibility_Category" column
df = df.withColumn("Visibility_Category",
    when(col("Item_Visibility") < 0.02, "Low Visibility")
    .when(col("Item_Visibility") < 0.10, "Medium Visibility")
    .otherwise("High Visibility")
)

# 3c. Round Item_Weight to 2 decimal places for consistency
df = df.withColumn("Item_Weight", round(col("Item_Weight"), 2))

# 3d. Aggregate: total sales & average MRP per Item_Type
agg_by_type = (df
    .groupBy("Item_Type")
    .agg(
        _sum("Item_Outlet_Sales").alias("Total_Sales"),
        round(avg("Item_MRP"), 2).alias("Avg_MRP"),
        count("Item_Identifier").alias("Item_Count"),
    )
    .orderBy(desc("Total_Sales"))
)

print("✅ STEP 3 — New columns added, aggregated by Item_Type.")
agg_by_type.display()

# ---------------------------------------------------------------------------
# STEP 4: ENRICH — Join aggregate back to original rows + window function
# ---------------------------------------------------------------------------
# 4a. Join the per-category totals back so every row knows its category total
df_enriched = df.join(agg_by_type, on="Item_Type", how="left")

# 4b. Sales contribution: how much % each row adds to its Item_Type total
df_enriched = df_enriched.withColumn(
    "Sales_Contribution_Pct",
    round((col("Item_Outlet_Sales") / col("Total_Sales")) * 100, 2)
)

# 4c. Window function: rank items within each Item_Type by sales (Top seller = 1)
from pyspark.sql.window import Window

window_spec = Window.partitionBy("Item_Type").orderBy(desc("Item_Outlet_Sales"))
df_enriched = df_enriched.withColumn("Sales_Rank_In_Category", 
    F.row_number().over(window_spec)
)

print("✅ STEP 4 — Data enriched with category totals, contribution %, and ranking.")
df_enriched.display()

# ---------------------------------------------------------------------------
# STEP 5: LOAD — Write curated dataset to destination
# ---------------------------------------------------------------------------
# Write to a CSV folder (overwrite if it already exists)
# coalesce(1) merges all partitions into a single output file for easy download
(df_enriched
    .coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("/Volumes/workspace/default/my_volume/curated_output/")
)

print("✅ STEP 5 — Curated dataset written to /Volumes/workspace/default/my_volume/curated_output/")
print("\n🎉 Pipeline complete! Review each step above to see how raw CSV data became analysis-ready.")
print("""
📚 QUICK RECAP of Data Engineering basics used here:
├─ Extract   : spark.read.format('csv')  → load raw data
├─ Clean     : dropDuplicates(), fillna(), when() → fix quality issues
├─ Transform : withColumn(), groupBy().agg(), round() → derive & summarize
├─ Enrich    : join(), window functions (row_number) → combine & rank
└─ Load      : write.mode('overwrite').csv() → save curated output
""")

In [0]:
# ============================================================================
# 🔗 Joins in PySpark — BigMart Sales Examples
# ============================================================================
# Joins combine two DataFrames based on a common column (or columns).
# Common join types:
#   inner  → only matching rows from both sides
#   left   → all rows from left DF, matched + nulls from right
#   right  → all rows from right DF, matched + nulls from left
#   outer  → all rows from both, nulls where no match
#   left_semi → rows from left that HAVE a match (no right columns)
#   left_anti  → rows from left that have NO match (no right columns)
# ============================================================================

from pyspark.sql.functions import col, sum, avg, round

# ---- Load data ----
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)

# ---- Create two small DataFrames to join ----

# Left DF: aggregate total sales per Item_Type
df_sales = (df.groupBy("Item_Type")
    .agg(sum("Item_Outlet_Sales").alias("Total_Sales"))
)

# Right DF: aggregate average MRP per Item_Type
df_mrp = (df.groupBy("Item_Type")
    .agg(round(avg("Item_MRP"), 2).alias("Avg_MRP"))
)

# ---- INNER JOIN: only Item_Types present in both DFs ----
df_inner = df_sales.join(df_mrp, on="Item_Type", how="inner")
print("🔹 INNER JOIN")
df_inner.display()

# ---- LEFT JOIN: all Item_Types from df_sales; nulls if no match in df_mrp ----
df_left = df_sales.join(df_mrp, on="Item_Type", how="left")
print("🔹 LEFT JOIN")
df_left.display()

# ---- RIGHT JOIN: all Item_Types from df_mrp; nulls if no match in df_sales ----
df_right = df_sales.join(df_mrp, on="Item_Type", how="right")
print("🔹 RIGHT JOIN")
df_right.display()

# ---- FULL OUTER JOIN: all Item_Types from both sides ----
df_outer = df_sales.join(df_mrp, on="Item_Type", how="outer")
print("🔹 FULL OUTER JOIN")
df_outer.display()

# ---- LEFT SEMI JOIN: Item_Types from df_sales that HAVE a match in df_mrp ----
# (returns only left columns — like a filtered INNER)
df_semi = df_sales.join(df_mrp, on="Item_Type", how="left_semi")
print("🔹 LEFT SEMI JOIN (only rows in left that have a match)")
df_semi.display()

# ---- LEFT ANTI JOIN: Item_Types from df_sales that have NO match in df_mrp ----
df_anti = df_sales.join(df_mrp, on="Item_Type", how="left_anti")
print("🔹 LEFT ANTI JOIN (only rows in left that have NO match)")
df_anti.display()

# ---- Join on MULTIPLE columns ----
df_multi = df.join(df_mrp, on=["Item_Type"], how="inner")
print("🔹 JOIN on multiple columns (Item_Type)")
df_multi.display()

# Explanation:
# - join(right_df, on=..., how=...) merges two DataFrames.
# - on= can be a single column name (string) or a list of column names.
# - how= controls the join type: inner, left, right, outer, left_semi, left_anti.
# - left_semi returns only left-side rows that match (no right columns).
# - left_anti returns only left-side rows that do NOT match (no right columns).
# - For large tables, use broadcast() on the smaller DF to optimize:
#     from pyspark.sql.functions import broadcast
#     df_sales.join(broadcast(df_mrp), on="Item_Type", how="inner")


In [0]:
df.select(col("Item_MRP").alias("MRP"), col("Item_Identifier")).filter(col("Item_MRP")>= 10).withColumnRenamed("Item_Identifier", "Items").display()

In [0]:
# ============================================================================
# 🛠️  Everyday Data Engineering Transformations with PySpark
# ============================================================================
# This cell walks through all the core transformations a Data Engineer uses
# on a daily basis, applied to the BigMart Sales dataset.
# Each section is labeled so you can run and observe step-by-step.
# ============================================================================

from pyspark.sql.functions import (
    col, when, round, lit, concat, substring, upper, lower,
    sum as _sum, avg, count, max as _max, min as _min, desc, asc
)
from pyspark.sql.types import DoubleType

# ------------------------------------------------------------------
# 1. LOAD / READ  —  Extract raw data from source
# ------------------------------------------------------------------
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)
print(" 2. LOAD — Raw data loaded | Rows:", df.count())
df.display()

# ------------------------------------------------------------------
# 2. SELECT  —  Pick only the columns you need
# ------------------------------------------------------------------
df_selected = df.select(
    "Item_Identifier", "Item_Type", "Item_MRP",
    "Item_Weight", "Item_Outlet_Sales", "Outlet_Identifier"
)
print("3  SELECT — Chose 6 columns")
df_selected.display(5)

# ------------------------------------------------------------------
# 3. FILTER / WHERE  —  Keep rows that meet a condition
# ------------------------------------------------------------------
df_filtered = df_selected.filter((col("Item_MRP") > 50) & (col("Item_Outlet_Sales") > 1000))
print("3  FILTER — MRP > 50 AND Sales > 1000 | Rows:", df_filtered.count())
df_filtered.display(5)

# ------------------------------------------------------------------
# 4. DROP DUPLICATES  —  Remove duplicate rows
# ------------------------------------------------------------------
df_dedup = df_filtered.dropDuplicates(["Item_Identifier", "Outlet_Identifier"])
print("4  DROP DUPLICATES — By Item_Identifier + Outlet_Identifier | Rows:", df_dedup.count())

# ------------------------------------------------------------------
# 5. FILL NULLS  —  Handle missing values
# ------------------------------------------------------------------
df_filled = df_dedup.fillna({"Item_Weight": 0})
print("5  FILL NULLS — Item_Weight nulls → 0")

# ------------------------------------------------------------------
# 6. WITH COLUMN (derive)  —  Add new calculated columns
# ------------------------------------------------------------------
df_derived = (df_filled
    .withColumn("Sales_Rounded", round(col("Item_Outlet_Sales"), 2))
    .withColumn("MRP_Category",
        when(col("Item_MRP") < 80, "Low")
        .when(col("Item_MRP") < 150, "Medium")
        .otherwise("High")
    )
    .withColumn("Weight_Category",
        when(col("Item_Weight") > 10, "Heavy")
        .when(col("Item_Weight") > 0, "Light")
        .otherwise("Unknown")
    )
)
print("6  WITH COLUMN — Added Sales_Rounded, MRP_Category, Weight_Category")
df_derived.display(5)

# ------------------------------------------------------------------
# 7. WITH COLUMN RENAMED  —  Rename existing columns
# ------------------------------------------------------------------
df_renamed = df_derived.withColumnRenamed("Item_Outlet_Sales", "Outlet_Sales")
print("7  RENAME — Item_Outlet_Sales → Outlet_Sales")

# ------------------------------------------------------------------
# 8. DROP COLUMNS  —  Remove columns not needed
# ------------------------------------------------------------------
df_dropped = df_renamed.drop("Item_Weight")  # already bucketed into Weight_Category
print("8  DROP COLUMN — Removed Item_Weight (replaced by Weight_Category)")

# ------------------------------------------------------------------
# 9. GROUP BY + AGG  —  Aggregate metrics per category
# ------------------------------------------------------------------
df_agg = (df_dropped
    .groupBy("Item_Type")
    .agg(
        _sum("Outlet_Sales").alias("Total_Sales"),
        round(avg("Item_MRP"), 2).alias("Avg_MRP"),
        count("Item_Identifier").alias("Item_Count"),
        _max("Outlet_Sales").alias("Max_Sale"),
        _min("Outlet_Sales").alias("Min_Sale"),
    )
)
print("9  GROUP BY + AGG — Total/Avg/Count/Max/Min per Item_Type")
df_agg.display()

# ------------------------------------------------------------------
# 10. ORDER BY / SORT  —  Sort by one or more columns
# ------------------------------------------------------------------
df_sorted = df_agg.orderBy(desc("Total_Sales"), asc("Avg_MRP"))
print("10  SORT — By Total_Sales DESC, Avg_MRP ASC")
df_sorted.display()

# ------------------------------------------------------------------
# 11. JOIN  —  Combine two DataFrames on a common key
# ------------------------------------------------------------------
# Small summary to join back onto the main DataFrame
df_outlet_summary = (df_dropped
    .groupBy("Outlet_Identifier")
    .agg(_sum("Outlet_Sales").alias("Outlet_Total_Sales"))
)

df_joined = df_dropped.join(df_outlet_summary, on="Outlet_Identifier", how="left")
print("11  JOIN — Added Outlet_Total_Sales per outlet")
df_joined.display(5)

# ------------------------------------------------------------------
# 12. DISTINCT  —  Get unique values
# ------------------------------------------------------------------
df_distinct = df_dropped.select("Item_Type").distinct().orderBy("Item_Type")
print("12 DISTINCT — Unique Item_Type values")
df_distinct.display()

# ------------------------------------------------------------------
# 13. UNION / UNION BY NAME  —  Stack two DataFrames vertically
# ------------------------------------------------------------------
df_part1 = df_dropped.filter(col("MRP_Category") == "Low")
df_part2 = df_dropped.filter(col("MRP_Category") == "High")
df_unioned = df_part1.unionByName(df_part2)
print("13  UNION — Stacked Low + High MRP rows | Rows:", df_unioned.count())

# ------------------------------------------------------------------
# 14. WITH COLUMN (string ops)  —  Text transformations
# ------------------------------------------------------------------
df_text = (df_unioned
    .withColumn("Item_ID_Upper", upper(col("Item_Identifier")))
    .withColumn("Item_ID_Prefix", substring(col("Item_Identifier"), 1, 3))
    .withColumn("Full_Label",
        concat(col("Item_Identifier"), lit(" - "), col("Item_Type")))
)
print("14  STRING OPS — upper, substring, concat on Item_Identifier")
df_text.select("Item_Identifier", "Item_ID_Upper", "Item_ID_Prefix", "Full_Label").display(5)

# ------------------------------------------------------------------
# 15. CAST  —  Change column data types
# ------------------------------------------------------------------
df_casted = df_text.withColumn("Item_MRP_Double", col("Item_MRP").cast(DoubleType()))
print("15  CAST — Item_MRP cast to DoubleType")

# ------------------------------------------------------------------
# 16. WRITE / SAVE  —  Persist the curated result
# ------------------------------------------------------------------
(df_casted
    .coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("/Volumes/workspace/default/my_volume/everyday_transformations_output/")
)
print("16  WRITE — Curated dataset saved to everyday_transformations_output/")

print("""
🎉  DONE!  Here's the recap of everyday transformations covered:
┌────┬─────────────────────┬────────────────────────────────────────────┐
│ #  │ Transformation       │ PySpark Method                             │
├────┼─────────────────────┼────────────────────────────────────────────┤
│ 1  │ Load / Read          │ spark.read.format('csv')                   │
│ 2  │ Select               │ df.select(...)                              │
│ 3  │ Filter / Where       │ df.filter(condition)                        │
│ 4  │ Drop Duplicates      │ df.dropDuplicates([...])                    │
│ 5  │ Fill Nulls           │ df.fillna({...})                            │
│ 6  │ With Column (derive) │ df.withColumn(name, expr)                   │
│ 7  │ With Column Renamed  │ df.withColumnRenamed(old, new)             │
│ 8  │ Drop Column          │ df.drop(col)                                │
│ 9  │ GroupBy + Aggregate  │ df.groupBy(...).agg(...)                    │
│ 10 │ Order By / Sort      │ df.orderBy(desc/asc)                        │
│ 11 │ Join                 │ df.join(other, on=, how=)                   │
│ 12 │ Distinct             │ df.select(...).distinct()                  │
│ 13 │ Union                │ df1.unionByName(df2)                        │
│ 14 │ String Operations    │ upper, substring, concat, lit               │
│ 15 │ Cast                 │ col(...).cast(Type)                         │
│ 16 │ Write / Save         │ df.write.mode('overwrite').csv(...)         │
└────┴─────────────────────┴────────────────────────────────────────────┘
""")

I have made this change to learn commit 

In [0]:
df.filter(col("Item_Weight").isNotNull() & (col("Item_Type") == "Baking Goods")).limit(10).display()

In [0]:
# ============================================================================
# 🎯 Data Engineering Interview Question (Beginner Level)
# Topic: CSV Loading, Cleaning, Aggregation & Ranking
# Scenario: "Find the Top 3 Item Types by Total Sales"
# ============================================================================
#
# Interview Prompt:
#   You are given a sales CSV file with columns like Item_Identifier,
#   Item_Type, Item_MRP, Item_Outlet_Sales, etc.
#
#   Write a PySpark pipeline that:
#     1. Reads the CSV with header and inferred schema.
#     2. Removes any duplicate rows.
#     3. Fills null values in Item_Weight with 0.
#     4. Groups by Item_Type and calculates:
#          - Total Sales (sum of Item_Outlet_Sales)
#          - Average MRP (rounded to 2 decimals)
#          - Item Count
#     5. Adds a rank column to find the Top 3 Item Types by Total Sales.
#     6. Displays the final result sorted by Total Sales descending.
#
# ============================================================================

from pyspark.sql.functions import col, sum as _sum, avg, round, count, desc, lit
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# ------------------------------------------------------------------
# STEP 1: Read raw CSV
# ------------------------------------------------------------------
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)
print("Raw row count:", df.count())

# ------------------------------------------------------------------
# STEP 2: Remove duplicates
# ------------------------------------------------------------------
df_clean = df.dropDuplicates()

# ------------------------------------------------------------------
# STEP 3: Fill nulls in Item_Weight with 0
# ------------------------------------------------------------------
df_clean = df_clean.fillna({"Item_Weight": 0})
# ------------------------------------------------------------------
# STEP 4: Group by Item_Type and aggregate
# ------------------------------------------------------------------

df_agg = df_clean.groupBy("Item_Type")\
.agg(_sum("Item_Outlet_Sales").alias("Total_Sales"),
        round(avg("Item_MRP"), 2).alias("MRP"),
        count("Item_Identifier").alias("count"),
    )
# ------------------------------------------------------------------
# STEP 5: Rank by Total Sales (descending) and pick Top 3
# ------------------------------------------------------------------

widow_func = Window.partitionBy(lit(1)).orderBy(desc("Total_Sales"))
ranked_df =df_agg.withColumn("Sales_Rank",F.row_number().over(widow_func))
top3 = ranked_df.filter(col("Sales_Rank") <=3)
# ------------------------------------------------------------------
# STEP 6: Display final result
# ------------------------------------------------------------------
print("Top 3 Item Types by Total Sales:")
top3.display()

# ============================================================================
# 💡 What the interviewer is looking for:
#   ✔ Can you read and infer schema from a CSV?
#   ✔ Do you handle duplicates and nulls before processing?
#   ✔ Can you write groupBy + agg correctly?
#   ✔ Do you know how to use window functions for ranking?
#   ✔ Is your code clean, readable, and well-commented?
# ============================================================================


In [0]:
# ============================================================================
# 🎯 Data Engineering Interview Question (Intermediate Level)
# Topic: Window Functions, Conditional Aggregation, Multi-Step Transforms
# Scenario: "Rank Outlets by Performance + Identify Top-Selling Item per Outlet"
# ============================================================================
#
# Interview Prompt:
#   Given the BigMart Sales CSV, write a PySpark pipeline that:
#     1. Reads the CSV, removes duplicates, and fills nulls.
#     2. Standardizes Item_Fat_Content (LF / low fat -> Low Fat, reg -> Regular).
#     3. For each Outlet_Identifier, calculates:
#          - Total Sales
#          - Average Sales
#          - Number of Unique Items Sold
#          - Total Sales from High MRP items only (Item_MRP >= 150)
#     4. Ranks outlets by Total Sales (dense rank, descending).
#     5. Uses a window function to find the TOP-selling Item_Type
#          within each outlet (by total sales for that item type).
#     6. Joins the outlet-level summary with the top-item-per-outlet result.
#     7. Displays the final enriched report sorted by outlet rank.
#
# ============================================================================

from pyspark.sql.functions import (
    col, when, round, sum as _sum, avg, countDistinct, desc, dense_rank, row_number
)
from pyspark.sql.window import Window

# ------------------------------------------------------------------
# STEP 1: Read + Clean + Standardize
# ------------------------------------------------------------------
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)

df = (df
    .dropDuplicates()
    .fillna({"Item_Weight": 0, "Outlet_Size": "Unknown"})
    .withColumn("Item_Fat_Content",
        when(col("Item_Fat_Content").isin("LF", "low fat"), "Low Fat")
        .when(col("Item_Fat_Content").isin("reg"), "Regular")
        .otherwise(col("Item_Fat_Content"))
    )
)
print("STEP 1 - Cleaned | Rows:", df.count())

# ------------------------------------------------------------------
# STEP 2: Outlet-level summary with conditional aggregation
# ------------------------------------------------------------------
outlet_summary = (df
    .groupBy("Outlet_Identifier")
    .agg(
        round(_sum("Item_Outlet_Sales"), 2).alias("Total_Sales"),
        round(avg("Item_Outlet_Sales"), 2).alias("Avg_Sales"),
        countDistinct("Item_Identifier").alias("Unique_Items"),
        round(_sum(when(col("Item_MRP") >= 150, col("Item_Outlet_Sales")).otherwise(0)), 2)
            .alias("High_MRP_Sales"),
    )
)
print("STEP 2 - Outlet summary with conditional aggregation")
outlet_summary.display()

# ------------------------------------------------------------------
# STEP 3: Rank outlets by Total Sales (dense rank)
# ------------------------------------------------------------------
rank_window = Window.orderBy(desc("Total_Sales"))
outlet_ranked = outlet_summary.withColumn("Outlet_Rank", dense_rank().over(rank_window))
print("STEP 3 - Outlets ranked by Total Sales")
outlet_ranked.display()

# ------------------------------------------------------------------
# STEP 4: Top-selling Item_Type per outlet (window function)
# ------------------------------------------------------------------
item_outlet_agg = (df
    .groupBy("Outlet_Identifier", "Item_Type")
    .agg(round(_sum("Item_Outlet_Sales"), 2).alias("Item_Type_Sales"))
)

top_item_window = Window.partitionBy("Outlet_Identifier").orderBy(desc("Item_Type_Sales"))
top_item_per_outlet = (item_outlet_agg
    .withColumn("Item_Rank", row_number().over(top_item_window))
    .filter(col("Item_Rank") == 1)
    .select("Outlet_Identifier", "Item_Type", "Item_Type_Sales")
    .withColumnRenamed("Item_Type", "Top_Item_Type")
    .withColumnRenamed("Item_Type_Sales", "Top_Item_Sales")
)
print("STEP 4 - Top-selling Item_Type per outlet identified")
top_item_per_outlet.display()

# ------------------------------------------------------------------
# STEP 5: Join outlet summary with top item per outlet
# ------------------------------------------------------------------
final_report = (outlet_ranked
    .join(top_item_per_outlet, on="Outlet_Identifier", how="left")
    .orderBy("Outlet_Rank")
    .select(
        "Outlet_Rank", "Outlet_Identifier", "Total_Sales", "Avg_Sales",
        "Unique_Items", "High_MRP_Sales", "Top_Item_Type", "Top_Item_Sales"
    )
)
print("STEP 5 - Final enriched outlet report")
final_report.display()

# ------------------------------------------------------------------
# STEP 6: Bonus - What % of each outlet's total sales comes from its top item type?
# ------------------------------------------------------------------
final_report = final_report.withColumn(
    "Top_Item_Sales_Pct",
    round((col("Top_Item_Sales") / col("Total_Sales")) * 100, 2)
)
print("STEP 6 - Added Top_Item_Sales_Pct (contribution of top item type)")
final_report.display()
df.display()

# ============================================================================
# What the interviewer is looking for (Intermediate level):
#   - Can you combine cleaning + standardization in a single chain?
#   - Do you know conditional aggregation inside agg()?  (sum(when(...)))
#   - Can you define and use multiple Window specs correctly?
#   - Do you understand dense_rank() vs row_number() and when to use each?
#   - Can you join aggregated results back together meaningfully?
#   - Is your pipeline readable, well-structured, and easy to follow?
# ============================================================================